In [0]:
%run ./Classroom-Setup-Common

In [0]:
-- ====================================================================
-- INSTRUCTOR-EDITABLE PARAMETERS
-- ====================================================================
-- Edit the catalog name and the MANAGED LOCATION below to match your own
-- UC external location. The path must be under an S3 / ADLS / GCS bucket
-- that YOU control, registered to UC via a storage credential. Databricks-
-- managed serverless storage (s3://dbstorage-prod-*) will not work because
-- external Iceberg engines cannot read those buckets.
-- ====================================================================

CREATE CATALOG IF NOT EXISTS instructor_interop_demo
  MANAGED LOCATION 's3://data-interoperability-with-unity-catalog-dev-metastore/unitycatalog/demo/instructor_interop_demo';

USE CATALOG instructor_interop_demo;

CREATE SCHEMA IF NOT EXISTS data_interoperability_tpcds;
USE SCHEMA data_interoperability_tpcds;

-- Sanity-check: confirm we are on the demo catalog before the CTASes run.
SELECT
  current_catalog() AS catalog,
  current_schema()  AS schema;

In [0]:
-- Native managed Iceberg, clustered for the same query patterns used in the
-- 2.2 Demo. CTAS sources directly from samples.tpcds_sf1000.store_sales so this
-- setup has no dependency on Module 2 / Instructor Demo Setup having run.
CREATE OR REPLACE TABLE store_sales_iceberg
USING ICEBERG
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
TBLPROPERTIES (
  'delta.enableDeletionVectors' = 'false',
  'delta.enableRowTracking'     = 'false'
)
AS SELECT * FROM samples.tpcds_sf1000.store_sales LIMIT 1000000;

In [0]:
BEGIN
  DECLARE actual_rows BIGINT;
  SET actual_rows = (SELECT COUNT(*) FROM store_sales_iceberg);

  IF actual_rows <> 1000000 THEN
    SELECT raise_error(
      'Table "' || current_catalog() || '.' || current_schema() || '.store_sales_iceberg" ' ||
      'has ' || CAST(actual_rows AS STRING) || ' rows, expected 1000000. ' ||
      'Drop the table and re-run this setup notebook.'
    );
  END IF;
END;

In [0]:
-- Delta + UniForm: the table is primarily Delta but UC also generates Iceberg
-- V3 metadata for it, so external Iceberg readers can load it through the same
-- REST endpoint as a native Iceberg table.
CREATE OR REPLACE TABLE store_sales_delta_uniform
CLUSTER BY (ss_sold_date_sk, ss_item_sk)
TBLPROPERTIES (
  'delta.enableIcebergCompatV3'          = 'true',
  'delta.universalFormat.enabledFormats' = 'iceberg'
)
AS SELECT * FROM samples.tpcds_sf1000.store_sales LIMIT 1000000;

In [0]:
BEGIN
  DECLARE actual_rows BIGINT;
  SET actual_rows = (SELECT COUNT(*) FROM store_sales_delta_uniform);

  IF actual_rows <> 1000000 THEN
    SELECT raise_error(
      'Table "' || current_catalog() || '.' || current_schema() || '.store_sales_delta_uniform" ' ||
      'has ' || CAST(actual_rows AS STRING) || ' rows, expected 1000000. ' ||
      'Drop the table and re-run this setup notebook.'
    );
  END IF;
END;

In [0]:
SELECT
  current_catalog() AS catalog,
  current_schema()  AS schema,
  (SELECT COUNT(*) FROM store_sales_iceberg)       AS iceberg_rows,
  (SELECT COUNT(*) FROM store_sales_delta_uniform) AS uniform_rows;